# Storing the data in a PostgreSQL database and SQL queries from Python

**This Jupyter notebook does the following:**

- Connects to a **PostgreSQL** database (`topic1`) from Python with the driver `psycopg2`
- Creates two tables with primary and foreign keys: `municipalities` and `apartments`
- Writes the enriched data from `Data/apartments_pre1990_enriched.csv` to the tables
- Creates an index and a view
- Runs SQL queries from Python, including PostgreSQL features (`FILTER`, window functions with `RANK() OVER`)
- Checks that the data in PostgreSQL are the same as in the SQLite database of `07_database_sqlite.ipynb`
- Closes the database connection

**Why a second database notebook?** `07_database_sqlite.ipynb` uses SQLite, which runs in the course environment without any installation; the following notebooks (08, 09) read from it. This notebook shows the same database in PostgreSQL, a client-server database system as used in companies.

**Before running this notebook**, install and start PostgreSQL once in the GitHub Codespace terminal:

```bash
bash Project_Topic1/setup_postgresql.sh
```

The script installs the PostgreSQL server, sets the password of the user `postgres`, creates the database `topic1` and installs the Python package `psycopg2-binary`. After a restart of the Codespace, start the server again with `sudo service postgresql start`.

## Libraries and settings

In [1]:
# Libraries
import os
import sqlite3
from decimal import Decimal
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

C:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Connect to the PostgreSQL database

The connection settings can be changed with environment variables (e.g. for another server); the default values fit the setup script.

In [2]:
conn = psycopg2.connect(host=os.environ.get('PGHOST', 'localhost'),
                        port=int(os.environ.get('PGPORT', 5432)),
                        dbname=os.environ.get('PGDATABASE', 'topic1'),
                        user=os.environ.get('PGUSER', 'postgres'),
                        password=os.environ.get('PGPASSWORD', 'postgres'))
cursor = conn.cursor()

cursor.execute('SELECT version()')
print(cursor.fetchone()[0])

PostgreSQL 17.2 on x86_64-windows, compiled by msvc-19.41.34123, 64-bit


### Function to run a query and return the result as a pandas data frame

PostgreSQL returns exact decimal numbers (`Decimal`) for `NUMERIC` columns; they are converted to normal numbers (`float`).

In [3]:
def query(sql):
    cursor.execute(sql)
    columns = [d[0] for d in cursor.description]
    result = pd.DataFrame(cursor.fetchall(), columns=columns)
    for c in result.columns:
        if result[c].map(lambda v: isinstance(v, Decimal)).any():
            result[c] = result[c].astype(float)
    return result

## Create the tables

In [4]:
# Delete old tables and the view, so that the notebook can be run several times
cursor.execute('DROP VIEW IF EXISTS v_apartments')
cursor.execute('DROP TABLE IF EXISTS apartments')
cursor.execute('DROP TABLE IF EXISTS municipalities')

cursor.execute('''CREATE TABLE municipalities (
                      bfs_number            INTEGER PRIMARY KEY,
                      bfs_name              VARCHAR(100) NOT NULL,
                      canton                CHAR(2),
                      language_region       VARCHAR(10),
                      population            INTEGER,
                      population_density    NUMERIC(10,2),
                      foreigners_pct        NUMERIC(5,2),
                      employees             INTEGER,
                      mean_taxable_income   NUMERIC(12,2))''')

cursor.execute('''CREATE TABLE apartments (
                      apartment_id          INTEGER PRIMARY KEY,
                      source                VARCHAR(20),
                      listing_url           VARCHAR(300) UNIQUE,
                      address               VARCHAR(200),
                      zip                   INTEGER,
                      locality              VARCHAR(100),
                      bfs_number            INTEGER REFERENCES municipalities(bfs_number),
                      lat                   NUMERIC(9,6),
                      lon                   NUMERIC(9,6),
                      rooms                 NUMERIC(4,1),
                      area                  NUMERIC(6,1),
                      price                 NUMERIC(8,2),
                      price_net             NUMERIC(8,2),
                      rent_per_m2           NUMERIC(6,2),
                      rent_net_per_m2       NUMERIC(6,2),
                      year_built            INTEGER,
                      year_renovated        INTEGER,
                      building_age          INTEGER,
                      furnished             BOOLEAN,
                      temporary             BOOLEAN,
                      renovation_group      VARCHAR(30),
                      reno_complete         BOOLEAN,
                      reno_kitchen          BOOLEAN,
                      reno_bathroom         BOOLEAN,
                      reno_floors           BOOLEAN,
                      reno_paint            BOOLEAN,
                      energy_windows        BOOLEAN,
                      energy_heating        BOOLEAN,
                      energy_insulation     BOOLEAN,
                      energy_roof           BOOLEAN,
                      energy_solar          BOOLEAN,
                      energy_minergie       BOOLEAN,
                      renovated_parts       VARCHAR(200))''')
conn.commit()

query('''SELECT table_name FROM information_schema.tables
         WHERE table_schema = 'public' ORDER BY table_name''')

,table_name
0,apartments
1,municipalities


## Write the data to the tables

Unlike SQLite, PostgreSQL has a real `BOOLEAN` data type, so the True/False columns are stored as booleans. Missing values (`NaN`) are written as `NULL`. `execute_values()` inserts all rows of a table with one SQL command.

In [5]:
df = pd.read_csv('./Data/apartments_pre1990_enriched.csv', sep=',', encoding='utf-8')
print('Dimension:', df.shape)

def to_rows(data):
    # Convert a data frame to a list of tuples with Python values and None for missing values
    data = data.astype(object).where(data.notna(), None)
    return [tuple(r) for r in data.values.tolist()]

# Table municipalities
mun_cols = ['bfs_number', 'bfs_name', 'canton', 'language_region', 'population', 'population_density',
            'foreigners_pct', 'employees', 'mean_taxable_income']
df_mun = df[mun_cols].drop_duplicates(subset='bfs_number').sort_values('bfs_number')
df_mun['employees'] = df_mun['employees'].astype('Int64')
execute_values(cursor, f"INSERT INTO municipalities ({', '.join(mun_cols)}) VALUES %s", to_rows(df_mun))

# Table apartments
df_apt = df.rename(columns={'area_clean': 'area',
                            'year_built_clean': 'year_built',
                            'year_renovated_clean': 'year_renovated'})
df_apt.insert(0, 'apartment_id', range(1, len(df_apt) + 1))
for c in ['year_built', 'year_renovated', 'building_age']:
    df_apt[c] = df_apt[c].astype('Int64')

cursor.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'apartments' ORDER BY ordinal_position")
apt_cols = [r[0] for r in cursor.fetchall()]
execute_values(cursor, f"INSERT INTO apartments ({', '.join(apt_cols)}) VALUES %s", to_rows(df_apt[apt_cols]))
conn.commit()

print('Rows in municipalities:', query('SELECT COUNT(*) AS n FROM municipalities')['n'][0])
print('Rows in apartments:', query('SELECT COUNT(*) AS n FROM apartments')['n'][0])

Dimension: (726, 50)
Rows in municipalities: 332
Rows in apartments: 726


## Index and view

- An **index** on `renovation_group` makes queries that filter or group by this column faster (with 726 rows this is not needed, but it is good practice for larger data).
- A **view** is a saved query: `v_apartments` joins the two tables, so that later queries do not have to repeat the `JOIN`.

In [6]:
cursor.execute('CREATE INDEX idx_apartments_group ON apartments (renovation_group)')
cursor.execute('''CREATE VIEW v_apartments AS
                  SELECT a.*, m.bfs_name, m.canton, m.language_region, m.population,
                         m.population_density, m.foreigners_pct, m.employees, m.mean_taxable_income
                  FROM apartments AS a
                  JOIN municipalities AS m ON a.bfs_number = m.bfs_number''')
conn.commit()

query('SELECT apartment_id, locality, bfs_name, canton, rooms, area, price, renovation_group FROM v_apartments LIMIT 5')

,apartment_id,locality,bfs_name,canton,rooms,area,price,renovation_group
0,1,La Chaux-de-Fonds,La Chaux-de-Fonds,NE,4.5,83.0,1485.0,general renovation
1,2,Luterbach,Luterbach,SO,4.5,108.0,1990.0,general renovation
2,3,Lenzburg,Lenzburg,AG,4.5,80.0,1760.0,energy renovation
3,4,Uster,Uster,ZH,5.5,116.0,2220.0,not renovated
4,5,Fribourg,Fribourg,FR,2.0,52.0,1275.0,general renovation


## SQL queries from Python

### Query 1: number of apartments and mean rents per renovation group (`GROUP BY`)

In [7]:
query('''SELECT renovation_group,
                COUNT(*)                       AS n,
                ROUND(AVG(area), 1)            AS mean_area,
                ROUND(AVG(price), 0)           AS mean_gross_rent,
                ROUND(AVG(rent_per_m2), 2)     AS mean_gross_rent_m2,
                ROUND(AVG(rent_net_per_m2), 2) AS mean_net_rent_m2,
                COUNT(rent_net_per_m2)         AS n_net_rent
         FROM apartments
         GROUP BY renovation_group
         ORDER BY mean_net_rent_m2 DESC''')

,renovation_group,n,mean_area,mean_gross_rent,mean_gross_rent_m2,mean_net_rent_m2,n_net_rent
0,energy renovation,43,86.4,2091.0,24.68,23.37,30
1,general renovation,486,87.0,1997.0,23.97,21.09,350
2,not renovated,197,85.3,1882.0,23.55,20.24,174


### Query 2: what was renovated in the energy group? (`COUNT(*) FILTER (WHERE ...)`, PostgreSQL)

In [8]:
query('''SELECT COUNT(*) FILTER (WHERE energy_heating)    AS heating,
                COUNT(*) FILTER (WHERE energy_windows)    AS windows,
                COUNT(*) FILTER (WHERE energy_solar)      AS solar,
                COUNT(*) FILTER (WHERE energy_insulation) AS insulation,
                COUNT(*) FILTER (WHERE energy_roof)       AS roof,
                COUNT(*) FILTER (WHERE energy_minergie)   AS minergie,
                COUNT(*) FILTER (WHERE reno_kitchen)      AS kitchen,
                COUNT(*) FILTER (WHERE reno_bathroom)     AS bathroom,
                COUNT(*)                                  AS n
         FROM apartments
         WHERE renovation_group = 'energy renovation' ''')

,heating,windows,solar,insulation,roof,minergie,kitchen,bathroom,n
0,17,13,12,8,2,6,10,10,43


### Query 3: mean net rent per m² by language region and renovation group (view)

In [9]:
query('''SELECT language_region,
                renovation_group,
                COUNT(*)                       AS n,
                ROUND(AVG(rent_net_per_m2), 2) AS mean_net_rent_m2
         FROM v_apartments
         GROUP BY language_region, renovation_group
         ORDER BY language_region, renovation_group''')

,language_region,renovation_group,n,mean_net_rent_m2
0,French,energy renovation,6,22.30
1,French,general renovation,83,23.09
2,French,not renovated,58,24.06
3,German,energy renovation,34,24.20
4,German,general renovation,377,20.83
5,German,not renovated,129,18.74
6,Italian,energy renovation,3,19.09
7,Italian,general renovation,26,18.97
8,Italian,not renovated,10,17.02


### Query 4: cantons with at least 20 apartments (`JOIN`, `HAVING`)

In [10]:
query('''SELECT m.canton,
                COUNT(*)                                                      AS n_apartments,
                COUNT(*) FILTER (WHERE a.renovation_group = 'energy renovation') AS n_energy,
                ROUND(AVG(m.mean_taxable_income), 0)                          AS mean_taxable_income
         FROM apartments AS a
         JOIN municipalities AS m ON a.bfs_number = m.bfs_number
         GROUP BY m.canton
         HAVING COUNT(*) >= 20
         ORDER BY n_energy DESC, n_apartments DESC''')

,canton,n_apartments,n_energy,mean_taxable_income
0,BE,132,11,65651.0
1,ZH,55,5,87711.0
2,SG,67,4,63545.0
3,AG,45,4,71855.0
4,GR,26,4,66074.0
5,TI,39,3,67910.0
6,BS,37,1,83387.0
7,FR,36,1,67509.0
8,SO,34,1,66310.0
9,TG,30,1,66682.0


### Query 5: the most expensive apartment per language region (window function `RANK() OVER`, PostgreSQL)

A window function ranks the apartments **within** each language region without grouping the rows.

In [11]:
query('''SELECT language_region, rank, bfs_name, rooms, area, rent_net_per_m2, renovation_group
         FROM (SELECT language_region, bfs_name, rooms, area, rent_net_per_m2, renovation_group,
                      RANK() OVER (PARTITION BY language_region ORDER BY rent_net_per_m2 DESC) AS rank
               FROM v_apartments
               WHERE rent_net_per_m2 IS NOT NULL) AS ranked
         WHERE rank <= 3
         ORDER BY language_region, rank''')

,language_region,rank,bfs_name,rooms,area,rent_net_per_m2,renovation_group
0,French,1,Lausanne,2.5,43.0,50.00,general renovation
1,French,2,Lausanne,1.0,24.0,49.79,not renovated
2,French,3,Genève,2.0,49.0,48.78,not renovated
3,German,1,Engelberg,3.5,55.0,51.82,general renovation
4,German,2,Pontresina,1.5,40.0,50.00,general renovation
5,German,3,Zug,4.5,103.0,44.56,general renovation
6,Italian,1,Brione sopra Minusio,2.5,43.0,45.35,general renovation
7,Italian,2,Lugano,3.5,133.0,45.11,general renovation
8,Italian,3,Lugano,2.5,55.0,35.45,not renovated


### Query 6: the 10 municipalities with the highest mean taxable income in the data

In [12]:
query('''SELECT m.bfs_name, m.canton, ROUND(m.mean_taxable_income, 0) AS mean_taxable_income,
                COUNT(a.apartment_id) AS n_apartments
         FROM municipalities AS m
         JOIN apartments AS a ON a.bfs_number = m.bfs_number
         WHERE m.mean_taxable_income IS NOT NULL
         GROUP BY m.bfs_number
         ORDER BY m.mean_taxable_income DESC
         LIMIT 10''')

,bfs_name,canton,mean_taxable_income,n_apartments
0,Feusisberg,SZ,229182.0,2
1,Herrliberg,ZH,174166.0,2
2,Kilchberg (ZH),ZH,166467.0,2
3,Zollikon,ZH,164757.0,1
4,Zug,ZG,148340.0,1
5,Meilen,ZH,130712.0,1
6,Crassier,VD,127230.0,1
7,Biel-Benken,BL,124979.0,1
8,Bottmingen,BL,119782.0,2
9,Pully,VD,117983.0,1


## Check: same data as in the SQLite database

The number of apartments and the mean net rent per m² per renovation group must be identical in PostgreSQL and in the SQLite database of `07_database_sqlite.ipynb`.

In [13]:
pg = query('''SELECT renovation_group, COUNT(*) AS n, ROUND(AVG(rent_net_per_m2), 4) AS mean_net_rent_m2
              FROM apartments GROUP BY renovation_group ORDER BY renovation_group''')

conn_lite = sqlite3.connect('./Data/topic1_apartments.db')
lite = pd.read_sql_query('''SELECT renovation_group, COUNT(*) AS n, ROUND(AVG(rent_net_per_m2), 4) AS mean_net_rent_m2
                            FROM apartments GROUP BY renovation_group ORDER BY renovation_group''', con=conn_lite)
conn_lite.close()

check = pg.merge(lite, on='renovation_group', suffixes=('_postgresql', '_sqlite'))
print('Identical:', (check['n_postgresql'] == check['n_sqlite']).all()
      and np.allclose(check['mean_net_rent_m2_postgresql'], check['mean_net_rent_m2_sqlite']))
check

Identical: True


,renovation_group,n_postgresql,mean_net_rent_m2_postgresql,n_sqlite,mean_net_rent_m2_sqlite
0,energy renovation,43,23.3710,43,23.3710
1,general renovation,486,21.0910,486,21.0910
2,not renovated,197,20.2389,197,20.2389


## Close db connection (if open)

In [14]:
cursor.close()
conn.close()
print('Connection closed:', conn.closed != 0)

Connection closed: True


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 21:42:04
Python Version: 3.11.8
-----------------------------------
